[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlnjsh/rl-basics/blob/main/Section_7_continuous_observation_spaces.ipynb)

<div style="text-align:center">
    <h1>
        Continuous state spaces
    </h1>
</div>

<br>

<div style="text-align:center">
    In this notebook we will learn how to adapt tabular methods to continuous state spaces. We will do it with two methods:
    state aggregation and tile coding.
</div>

<br>

In [ ]:
# === Setup: load the shared course modules =====================================
# The long environment-definition cell has been moved into two files that live
# next to this notebook: `envs.py` (the Maze environment) and `utils.py` (all the
# plotting / evaluation helpers). That keeps every lesson focused on the algorithm.
import sys, os
if 'google.colab' in sys.modules and not os.path.exists('envs.py'):
    # On Google Colab there are no local files, so install the pinned gym version
    # and download the two helper modules from the course repository.
    !pip install -qq gym==0.23.0 pygame
    !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/envs.py
    !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/utils.py

from envs import Maze              # The 5x5 grid-world environment.
from utils import test_agent, plot_policy, plot_tabular_cost_to_go, plot_stats, seed_everything


## Import the necessary software libraries:

In [ ]:
import random
import gym
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

## Implement state aggregation


### Create the environment

### Why tabular methods fail on continuous state spaces

Until now we stored one number per state-action pair in a **table** - fine when states are a small finite set, like maze cells. MountainCar is different: its state is a pair of **real numbers** (position, velocity), so there are *infinitely many* states. We cannot build a row per state, and we would almost never revisit the exact same state, so the table would never learn.

The fix is **discretization (state aggregation)**: chop each continuous dimension into a finite number of **buckets**, and treat any two states in the same bucket as the same. Then ordinary tabular SARSA works again.

> **TODO (you):** In the cells below, (1) create the `MountainCar-v0` environment and seed it, then (2) write a `StateAggregationEnv` observation wrapper. Hint: `np.linspace` builds the bucket **edges** and `np.digitize` maps a continuous value to its bucket index. Finally instantiate it as `saenv` with a 20 x 20 grid.

### Create the state aggregation wrapper

### Compare the original environment to the one with aggregated states

In [ ]:
print(f"Modified observation space: {saenv.observation_space}, \n\
Sample state: {saenv.observation_space.sample()}")

In [ ]:
print(f"Original observation space: {env.observation_space}, \n\
Sample state: {env.observation_space.sample()}")

### Create the $Q(s,a)$ value table

In [ ]:
action_values = np.zeros((20,20, 3))

### Create the $\epsilon$-greedy policy: $\pi(s)$

In [ ]:
def policy(state, epsilon=0.):
    if np.random.random() < epsilon:
        return np.random.randint(3)
    else:
        av = action_values[state]
        return np.random.choice(np.flatnonzero(av == av.max()))

### Test the SARSA algorithm on the modified environment

In [ ]:
def sarsa(action_values, policy, episodes, alpha=0.1, gamma=0.99, epsilon=0.2):
    stats = {'Returns': []}
    for episode in tqdm(range(1, episodes + 1)):
        state = saenv.reset()
        action = policy(state, epsilon)
        done = False
        ep_return = 0
        while not done:
            next_state, reward, done, _ = saenv.step(action)
            next_action = policy(next_state, epsilon)

            qsa = action_values[state][action]
            next_qsa = action_values[next_state][next_action]
            action_values[state][action] = qsa + alpha * (reward + gamma * next_qsa - qsa)
            state = next_state
            action = next_action
            ep_return += reward
        stats['Returns'].append(ep_return)
    return stats

In [ ]:
stats = sarsa(action_values, policy, 20000, alpha=0.1, epsilon=0.)

In [ ]:
plot_stats(stats)

### Plot the learned policy: $\pi(s)$

In [ ]:
plot_policy(action_values, env.render(mode='rgb_array'), \
            action_meanings={0: 'B', 1: 'N', 2: 'F'})

### Plot the cost to go: $ - \max_a \hat q(s,a|\theta)$

In [ ]:
plot_tabular_cost_to_go(action_values, xlabel="Car Position", ylabel="Velocity")

### Test the resulting policy

In [ ]:
test_agent(saenv, policy, 2)

<br><br><br><br>

## Implement Tile Coding


### Create the environment

### Why tile coding beats a single coarse grid

State aggregation works, but a single grid has a weakness: every point inside a bucket looks **identical**, and a tiny move across a bucket edge looks **completely different**. A coarse grid loses detail; a fine grid needs a huge table.

**Tile coding** fixes this with several overlapping grids (*tilings*), each shifted by a fraction of a cell. A state activates one tile per tiling, and its value is the **average across all tilings**. Nearby points share most (not all) tiles, so the agent can still tell them apart while generalizing between them.

> **TODO (you):** Re-create the MountainCar environment, then build a `TileCodingEnv` observation wrapper that returns a *list* of bucket-index tuples - one per tiling. Think about how to shift each tiling so the grids do not line up.

### Create the Tile Coding wrapper

### Compare the original environment to the one with aggregated states

In [ ]:
print(f"Modified observation space: {tcenv.observation_space}, \n\
Sample state: {tcenv.reset()}")

In [ ]:
print(f"Original observation space: {env.observation_space}, \n\
Sample state: {env.reset()}")

### Create the $Q(s,a)$ value table

In [ ]:
action_values = np.zeros((4, 20, 20, 3))

### Create the $\epsilon$-greedy policy: $\pi(s)$

In [ ]:
def policy(state, epsilon=0.):
    if np.random.random() < epsilon:
        return np.random.randint(3)
    else:
        av_list = []
        for i, idx in enumerate(state):
            av = action_values[i][idx]
            av_list.append(av)

        av = np.mean(av_list, axis=0)
        return np.random.choice(np.flatnonzero(av==av.max()))

### Test the SARSA algorithm on the modified environment

In [ ]:
def sarsa(action_values, policy, episodes, alpha=0.1, gamma=0.99, epsilon=0.2):
    stats = {'Returns': []}
    for episode in tqdm(range(1, episodes + 1)):
        state = tcenv.reset()
        action = policy(state, epsilon)
        done = False
        ep_return = 0
        while not done:
            next_state, reward, done, _ = tcenv.step(action)
            next_action = policy(next_state, epsilon)

            for i, (idx, next_idx) in enumerate(zip(state, next_state)):
                qsa = action_values[i][idx][action]
                next_qsa = action_values[i][next_idx][next_action]
                action_values[i][idx][action] = qsa + alpha * (reward + gamma * next_qsa - qsa)

            state = next_state
            action = next_action
            ep_return += reward
        stats['Returns'].append(ep_return)
    return stats

In [ ]:
stats = sarsa(action_values, policy, 20000, alpha=0.1, epsilon=0.)

In [ ]:
plot_stats(stats)

### Plot the learned policy: $\pi(s)$

In [ ]:
plot_policy(action_values.mean(axis=0), env.render(mode='rgb_array'), \
            action_meanings={0: 'B', 1: 'N', 2: 'F'})

### Plot the cost to go: $ - \max_a \hat q(s,a|\theta)$

In [ ]:
plot_tabular_cost_to_go(action_values.mean(axis=0), \
                        xlabel="Car Position", ylabel="Velocity")

### Test the resulting policy

In [ ]:
test_agent(tcenv, policy, 2)

## Resources

[[1] Reinforcement Learning: An Introduction. Section 9.5.4: Tile Coding](https://web.stanford.edu/class/psych209/Readings/SuttonBartoIPRLBook2ndEd.pdf)